<a href="https://colab.research.google.com/github/Arpit11-svg/DL/blob/main/BERT_for_QA_%26_Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets -q

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import TrainingArguments, Trainer
import torch

In [ ]:
dataset = load_dataset("imdb")

train_data = dataset["train"].shuffle(seed=42).select(range(1000))
test_data = dataset["test"].shuffle(seed=42).select(range(200))

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"],
                     padding=True,
                     truncation=True)

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

In [ ]:
train_data.set_format("torch",
                      columns=["input_ids", "attention_mask", "label"])

test_data.set_format("torch",
                     columns=["input_ids", "attention_mask", "label"])

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=10
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data
)

trainer.train()

In [ ]:
predictions = trainer.predict(test_data)

preds = torch.argmax(
    torch.tensor(predictions.predictions),
    axis=1
)

labels = predictions.label_ids

accuracy = (preds == torch.tensor(labels)).float().mean()

print("Accuracy:", accuracy.item())